# GeoChat-7B Local Feasibility Experiment (Phase 6)

**Project:** SatQuery AI  —  **Phase:** 6 (feasibility experiment only)  
**Date:** 2026-09-10  —  **Machine:** local Windows laptop  
**Verdict:** `LOCAL_NOT_FEASIBLE` (analytical proof)

This notebook investigates whether `MBZUAI/geochat-7B` can realistically run locally.
It performs **no model-weight download, no ML package install, and no model inference**.
The conclusion is derived from measured hardware capacity + published model metadata
(the spec's own test flow in `docs/satquery-master-build-spec.md` §5.4 says to check
GPU/VRAM first and quantize only "if VRAM < 16GB"; here even quantized footprints
cannot fit).

> Data provenance: every number  is marked **MEASURED** (read from this machine or
> from the Hugging Face repository) or **ESTIMATE** (computed from documented
> parameters). No inference speed or memory figure was produced experimentally,
> because no model was ever loaded.


## 1. Environment & hardware (MEASURED)

Checks below are pure diagnostics: Python/platform introspection, OS global memory
query via the Windows API, and `nvidia-smi` parsing. No ML package is required.


In [ ]:
import platform
import struct
import sys

print("python_executable :", sys.executable)
print("python_version    :", sys.version.split()[0])
print("interpreter       :", " ".join(sys.version.split()[1:]))
print("platform          :", platform.platform())
print("machine           :", platform.machine())
print("pointer_bits      :", struct.calcsize("P") * 8)
for mod in ("torch", "transformers", "bitsandbytes", "accelerate", "peft"):
    try:
        __import__(mod)
        present = True
    except ImportError:
        present = False
    print(f"{mod:14s} installed: {present}")


python_executable : C:\Users\ihars\OneDrive\Desktop\satquery-ai\backend\venv\Scripts\python.exe
python_version    : 3.14.4
interpreter       : [MSC v.1944 64 bit (AMD64)]
platform          : Windows-11-10.0.26200-SP0
machine           : AMD64
pointer_bits      : 64
torch             installed: False
transformers      installed: False
bitsandbytes      installed: False
accelerate        installed: False
peft              installed: False


In [ ]:
import ctypes


class MEMORYSTATUSEX(ctypes.Structure):
    _fields_ = [
        ("dwLength", ctypes.c_ulong),
        ("dwMemoryLoad", ctypes.c_ulong),
        ("ullTotalPhys", ctypes.c_ulonglong),
        ("ullAvailPhys", ctypes.c_ulonglong),
        ("ullTotalPageFile", ctypes.c_ulonglong),
        ("ullAvailPageFile", ctypes.c_ulonglong),
        ("ullTotalVirtual", ctypes.c_ulonglong),
        ("ullAvailVirtual", ctypes.c_ulonglong),
        ("ullAvailExtendedVirtual", ctypes.c_ulonglong),
    ]


stat = MEMORYSTATUSEX()
stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat))
gib = 1024 ** 3
print("OS   : Microsoft Windows 11 Pro (build 26200)")
print(f"Total physical RAM : {stat.ullTotalPhys / gib:.2f} GiB   (MEASURED)")
print(f"Available physical: {stat.ullAvailPhys / gib:.2f} GiB   (MEASURED at run time)")
print(f"Memory load       : {stat.dwMemoryLoad}%")


OS   : Microsoft Windows 11 Pro (build 26200)
Total physical RAM : 63.69 GiB   (MEASURED)
Available physical: 46.23 GiB   (MEASURED at run time)
Memory load       : 27%


In [ ]:
import shutil
import subprocess
import sys

FIELDS = ["name", "memory.total", "memory.used", "driver_version", "compute_cap"]
exe = shutil.which("nvidia-smi")
if exe is None:
    print("nvidia-smi not found - no NVIDIA GPU detected.")
    sys.exit(0)
query = ",".join(FIELDS)
out = subprocess.run(
    [exe, "--query-gpu=" + query, "--format=csv,noheader"],
    capture_output=True, text=True, timeout=20, check=True,
).stdout.strip()
print(out)
print("Driver/CUDA context: n/a here - see tracked nvidia-smi full output below")


NVIDIA T600 Laptop GPU, 4096 MiB, 0 MiB, 597.06, 7.5
Driver/CUDA context: n/a here - see tracked nvidia-smi full output below


In [ ]:
import shutil
import subprocess
import sys

exe = shutil.which("nvidia-smi")
if exe is None:
    print("nvidia-smi not found.")
    sys.exit(0)
out = subprocess.run([exe], capture_output=True, text=True, timeout=20, check=True).stdout
head = [l for l in out.splitlines() if any(k in l for k in
        ("Driver Version", "CUDA Version", "NVIDIA-SMI"))]
print("\n".join(head))
with_disp = [l for l in out.splitlines() if "NVIDIA T600" in l or "|   0 " in l]
print("\n".join(with_disp[:2]))


NVIDIA-SMI 597.06                 Driver Version: 597.06         CUDA Version: 13.2
|   0  NVIDIA T600 Laptop GPU       WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   56C    P0             15W /   35W |       0MiB /   4096MiB |      0%      Default |


## 2. GeoChat-7B model metadata (MEASURED from Hugging Face)

Only small text/JSON files are fetched (no weights). Source: `MBZUAI/geochat-7B`
(model card + `config.json`). Facts captured 2026-09-10.

**Architecture:** `GeoChatLlamaForCausalLM` = LLaVA-1.5-7B (Vicuna v1.5 7B decoder +
CLIP ViT-L/14-336 vision tower at 504x504 high-res) — fine-tuned on
GeoChat_Instruct (318k remote-sensing instruction samples).


In [ ]:
import json
import urllib.request

BASE = "https://huggingface.co/MBZUAI/geochat-7B"


def fetch(path):
    with urllib.request.urlopen(f"{BASE}/raw/main/{path}", timeout=20) as r:
        return r.read().decode()


config = json.loads(fetch("config.json"))
api = json.loads(urllib.request.urlopen(BASE, timeout=20).read().decode())

print("== config.json (key fields, MEASURED from HF) ==")
for key in [
    "architectures", "model_type", "hidden_size", "num_hidden_layers",
    "num_attention_heads", "num_key_value_heads", "vocab_size",
    "max_position_embeddings", "mm_vision_tower", "mm_hidden_size",
    "mm_projector_type", "torch_dtype", "transformers_version",
]:
    print(f"  {key:26s}: {config.get(key)}")

print()
print("== HF repository (MEASURED) ==")
card = api.get("cardData", {}) or {}
print(f"  gated   : {api.get('gated')}")
print(f"  license : {card.get('license')}")
print("  weight files:")
for s in sorted(api.get("siblings", []), key=lambda x: x["rfilename"]):
    n = s["rfilename"]
    if n == "pytorch_model.bin.index.json" or n.startswith("pytorch_model-"):
        print(f"    - {n}")


== config.json (key fields, MEASURED from HF) ==
  architectures           : ['GeoChatLlamaForCausalLM']
  model_type              : geochat
  hidden_size             : 4096
  num_hidden_layers       : 32
  num_attention_heads     : 32
  num_key_value_heads     : 32
  vocab_size              : 32000
  max_position_embeddings : 4096
  mm_vision_tower         : openai/clip-vit-large-patch14-336
  mm_hidden_size          : 1024
  mm_projector_type       : mlp2x_gelu
  torch_dtype             : float16
  transformers_version    : 4.31.0

== HF repository (MEASURED) ==
  gated   : False
  license : apache-2.0
  weight files:
    - pytorch_model-00001-of-00002.bin
    - pytorch_model-00002-of-00002.bin
    - pytorch_model.bin.index.json


## 3. Weight-size evidence (MEASURED)

The two fp16 weight shards total **~14.2 GB** (decimal) per Hugging Face file listing,
and the parameter-based estimate independently lands within ~0.1 GB of that figure,
so the two agree.

| Evidence | Value | Kind |
|---|---|---|
| `pytorch_model-00001-of-00002.bin` | 9.98 GB (decimal) | MEASURED (HF) |
| `pytorch_model-00002-of-00002.bin` | 4.2 GB (decimal) | MEASURED (HF) |
| Sum | ≈ 14.2 GB decimal ≈ 13.2 GiB | MEASURED |
| fp16 params estimate (7.05e9 x 2 bytes) | ≈ 14.1 GB decimal ≈ 13.1 GiB | ESTIMATE |

LLM 6.738B params gives the bulk; CLIP ViT-L vision tower ~0.307B; MLP projector ~8M.


In [ ]:
GB = 1_000_000_000          # decimal gigabytes (matches HF file sizes)
GIB = 1024 ** 3             # binary gibibytes (matches hardware reporting)
LLM = 6.738e9
VISION = 0.307e9
PROJ = 0.0084e9
TOTAL = LLM + VISION + PROJ

def gb(n): return n / GB
def gib(n): return n / GIB

# KV cache: 2 (K,V) x 32 layers x 32 kv-heads x 128 head-dim x 4096 seq x 2 bytes
KV = 2 * 32 * 32 * 128 * 4096 * 2

print("Weight memory (no cache/execution overhead yet):")
print(f"  fp16 total          : {gb(TOTAL*2):.2f} GB dec / {gib(TOTAL*2):.2f} GiB   (ESTIMATE)")
print(f"  4-bit LLM + fp16 vy : {gib(LLM*0.5 + (VISION+PROJ)*2):.2f} GiB               (ESTIMATE)")
print(f"  KV cache @4096 tok  : {gib(KV):.2f} GiB               (ESTIMATE)")
print()
print("Pipeline footprints (weights + KV cache, no CUDA/WDDM context):")
fp16_pipe = gib(TOTAL*2) + gib(KV)
q4_pipe = gib(LLM*0.5 + (VISION+PROJ)*2) + gib(KV)
print(f"  fp16 pipeline       : {fp16_pipe:.2f} GiB   (ESTIMATE)")
print(f"  4-bit best case     : {q4_pipe:.2f} GiB   (ESTIMATE)")
print(f"  + CUDA ctx/act/DDM  : ~1.5-2 GiB extra  (ESTIMATE, typical for 7B fp16) ")
print()
print("Hardware capacity:")
print(f"  GPU VRAM            : 4096 MiB = 4.00 GiB   (MEASURED, nvidia-smi)")
print(f"  System RAM          : 63.69 GiB total       (MEASURED, Win API)")
print()
print("Result on this machine's GPU:")
print(f"  fp16 overshoots    : +{fp16_pipe-4.0:5.2f} GiB")
print(f"  4-bit overshoots   : +{q4_pipe-4.0:5.2f} GiB  (before the ~1.5-2 GiB context -> OOM)")


Weight memory (no cache/execution overhead yet):
  fp16 total          : 14.11 GB dec / 13.14 GiB   (ESTIMATE)
  4-bit LLM + fp16 vy : 3.73 GiB               (ESTIMATE)
  KV cache @4096 tok  : 2.00 GiB               (ESTIMATE)

Pipeline footprints (weights + KV cache, no CUDA/WDDM context):
  fp16 pipeline       : 15.14 GiB   (ESTIMATE)
  4-bit best case     : 5.73 GiB   (ESTIMATE)
  + CUDA ctx/act/DDM  : ~1.5-2 GiB extra  (ESTIMATE, typical for 7B fp16) 

Hardware capacity:
  GPU VRAM            : 4096 MiB = 4.00 GiB   (MEASURED, nvidia-smi)
  System RAM          : 63.69 GiB total       (MEASURED, Win API)

Result on this machine's GPU:
  fp16 overshoots    : +11.14 GiB
  4-bit overshoots   : +1.73 GiB  (before the ~1.5-2 GiB context -> OOM)


## 4. Python / Windows / CUDA compatibility considerations

- **Python 3.14.4 (MEASURED)** is **not** by itself the blocker:
  PyTorch 2.9 has Python-3.14 compatibility, and bitsandbytes has shipped a
  Python-3.14 fix against PyTorch 2.9 (upstream release notes), including
  Windows x86-64 CUDA wheels (SM60+; SM75 recommended — the T600 is SM75).
- The **real secondary blocker** is the model's `transformers_version: 4.31.0`
  (MEASURED). The LLaVA-1.5 remote code (`trust_remote_code=True`) imports APIs
  that modern transformers removed; pinning the old transformers line on
  Python 3.14 / Windows would require source builds that are not published and
  are impractical to produce here.
- T600 is a low-power **laptop** part (TGP 35 W), WDDM driver model, with CUDA
  13.2 driver present (MEASURED). Even with a perfect software stack, the 4 GiB
  VRAM ceiling is the decisive constraint (Section 3).

Net: the software stack *could* in principle be assembled, but doing so is
pointless because the hardware cannot hold the quantized pipeline.


## 5. Why empirical install/inference was intentionally NOT performed

The analytical proof in Sections 2-3 already decides the question:

1. **fp16** needs ~13.2 GiB of weights alone; the GPU has **4.00 GiB**. No path.
2. **4-bit best case** (quantized LLM + fp16 vision tower) is ~5.73 GiB *before*
   the ~1.5-2 GiB CUDA context/activation/Windows-driver overhead — already over
   budget at load time; inference would immediately OOM.
3. A **CPU-offload** run (fits the 63.7 GB system RAM) would satisfy the spec's
   load-once + "answers within a couple of minutes" only in the loosest sense;
   expected CPU decode on a 6.7B model is seconds per token — per-request
   latency on the order of minutes. The spec explicitly rejects that as the
   fallback trigger ("If it loads and answers within a couple minutes, use
   Path A; otherwise use Path B").

Per the Phase 6 safety rules, a multi-GB weight download (~14.2 GB) plus a
torch/bitsandbytes install (~2-3 GB) is not justified when the outcome is known
to be OOM. **No inference was attempted, and no inference numbers are reported.**

This conclusion is therefore **derived**, not simulated or fabricated.


## 6. Swappable VQA provider architecture (design note, not implemented)

`docs/satquery-master-build-spec.md` §5.4 requires a single `answer_question()`
interface behind a `VQA_BACKEND=geochat|api` flag. The recommended structure keeps
the router and schema unchanged:

```text
answer_question(image_path, query_text)        # VQAService facade, vqa.py
   |
   +-- LocalGeoChatProvider      # VQA_BACKEND=geochat (future: 4-bit + load-once)
   +-- HostedVLMProvider          # VQA_BACKEND=api     (RS-engineered system prompt
   |                                + few-shot satellite Q&A, self-reported confidence)
   +-- FutureRemoteSensingVLMProvider   # any VLM fine-tuned on BigEarthNet/VRSBench
```

- Selection via environment flag only; callers never know which path ran.
- `execution_status` stays truthful (`not_implemented` today; `completed` only
  when a real model answers).
- No API keys or credentials added in this phase; hosted provider is **not**
  built because Phase 6 forbids implementing the hosted API and adding secrets.

Nothing in this notebook changes `vqa.py`, the API, the schema, or the database.


## 7. Dataset / fine-tuning findings (BigEarthNet & VRSBench)

**Remote-sensing adaptation is MANDATORY** per the official problem statement:

> "At least one visual or vision-language component must be fine-tuned or
> otherwise adapted using BigEarthNet.txt or any open source training data."

- `BigEarthNet.txt` is explicitly named as an **allowed / primary adaptation
  source** in the problem statement; other open-source training data is also
  permitted by the wording.
- The requirement is satisfied by adapting at least **one** visual/VLM
  component — **fine-tuning GeoChat-7B specifically is NOT required.**
- Candidate open data for that adaptation: BigEarthNet (590k multi-label
  Sentinel-1/2 patches) and VRSBench (~29k images / ~272k question-answer pairs),
  suitable for few-shot prompting or LoRA-style adaptation of a smaller vision
  model.
- **Hardware reality:** adapting a 7B VLM (GeoChat's own SFT took ~25 h on
  3x A100-40 GB) is impractical on a 4 GiB laptop GPU; adaptation of the
  mandatory visual component/s is FUTURE WORK, plausibly via the hosted-VLM
  path (Path B) with an RS-engineered prompt, or rented GPU time later.
- **Nothing has been adapted yet.** No dataset was downloaded, no fine-tuning or
  adaptation was run, and this notebook makes no claim that any adaptation has
  been performed. Phase 6 only settles the GeoChat local-inference feasibility
  question.


In [ ]:
VRAM_GIB = 4.00                     # MEASURED (nvidia-smi)
BEST_4BIT_PIPE_GIB = 5.73           # ESTIMATE (Section 3: weights + KV cache)
BEST_4BIT_WITH_OVERHEAD = BEST_4BIT_PIPE_GIB + 1.5  # +min. CUDA/WDDM overhead

verdict = (
    "LOCAL_NOT_FEASIBLE"
    if BEST_4BIT_WITH_OVERHEAD > VRAM_GIB
    else "LOCAL_FEASIBLE"
)

print("Final classification:", verdict)
print()
print(f"GPU VRAM (MEASURED)            : {VRAM_GIB:.2f} GiB")
print(f"Best-case 4-bit pipeline (est) : {BEST_4BIT_PIPE_GIB:.2f} GiB")
print(f"With minimal ~1.5 GiB overhead : {BEST_4BIT_WITH_OVERHEAD:.2f} GiB")
print()
print("Reason: a 4 GiB NVIDIA T600 cannot hold even the most economical")
print("quantized GeoChat pipeline; fp16 requires ~3.3x the available VRAM.")
print("Software stack is installed in no environment and inference was never")
print("attempted - the verdict follows from measured capacity + published size.")


Final classification: LOCAL_NOT_FEASIBLE

GPU VRAM (MEASURED)            : 4.00 GiB
Best-case 4-bit pipeline (est) : 5.73 GiB
With minimal ~1.5 GiB overhead : 7.23 GiB

Reason: a 4 GiB NVIDIA T600 cannot hold even the most economical
quantized GeoChat pipeline; fp16 requires ~3.3x the available VRAM.
Software stack is installed in no environment and inference was never
attempted - the verdict follows from measured capacity + published size.


## Verdict summary

**`LOCAL_NOT_FEASIBLE`** — GeoChat-7B cannot run locally on this machine.

- Measured hardware: NVIDIA T600 Laptop, 4096 MiB VRAM; 63.7 GB system RAM.
- Measured model: ~14.2 GB of fp16 checkpoints (9.98 + 4.2 GB shards),
  transformers 4.31, LLaVA-1.5 architecture, apache-2.0, public.
- Even 4-bit-quantized the pipeline needs ~5.7 GiB before execution overhead
  on a 4.0 GiB GPU -> guaranteed OOM at load.
- CPU offload is not a workable "couple-of-minutes" path per spec §5.4.
- Python 3.14 is not the blocker; the old transformers remote-code line plus
  VRAM is.

**Recommended direction (later phases):** build the swappable
`VQAService -> HostedVLMProvider` (Path B) with an RS-engineered prompt and
few-shot examples; keep `LocalGeoChatProvider` as a future option for rented/
server hardware; keep every "VQA" answer honest (`execution_status`) until a
real model runs. No code changes were made by this experiment.

**Mandatory RS adaptation is FUTURE WORK** (nothing has ever been adapted; see
Section 7 for the exact problem-statement requirement).
